In [83]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import math

In [69]:
class stockLSTM(nn.Module):
    def __init__(self, num_features=6, hidden_size=6, history_window=12):
        super().__init__()
    
        self.num_features = num_features
        self.hidden_size = hidden_size
        self.history_window = history_window
    
        self.lstm = nn.LSTM(
            input_size = self.num_features,
            hidden_size = self.hidden_size,
            batch_first=True
        )

    def forward(self, x):
        """
        (n_stocks, history_window, n_features) -> (n_stocks, history_window, hidden_size)
        """
        if x.shape[1] != self.history_window:
            raise ValueError(
                f"Expected history_window = {self.history_window}, got {x.shape[1]}"
            )
            
        h_all, _ = self.lstm(x)
        return h_all

In [72]:
class historyAttention(nn.Module):
    def __init__(self, hidden_size = 6):
        super().__init__()

        self.W1 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W2 = nn.Linear(hidden_size, hidden_size, bias=False)
        self.w = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, h_all):
        """
        (n_stocks, history_window, hidden_size) -> (n_stocks, hidden_size)
        """
        h_last = h_all[:, -1, :] # (n_stocks, hidden_size)
        W1h = self.W1(h_all) # (n_stocks, history_window, hidden_size)
        W2h = self.W2(h_last).unsqueeze(1) # (n_stocks, 1, hidden_size)

        alpha = self.w(torch.tanh(W1h + W2h)).squeeze(-1) # (n_stocks, history_window)
        att = torch.softmax(alpha, dim=1) # (n_stocks, history_window)
        r = torch.sum(att.unsqueeze(-1) * h_all, dim=1) # (n_stocks, hidden_size)

        return r

In [93]:
class crossAssetAttention(nn.Module):
    def __init__(self, hidden_size = 6):
        super().__init__()

        self.Wq = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Wk = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Wv = nn.Linear(hidden_size, hidden_size, bias=False)

        self.out = nn.Linear(hidden_size, 1)

    def forward(self, r):
        """
        (n_stocks, hidden_size) -> (n_stocks)
        """
        Q = self.Wq(r) # (n_stocks, hidden_size)
        K = self.Wk(r) # (n_stocks, hidden_size)
        V = self.Wv(r) # (n_stocks, hidden_size)

        scores = torch.matmul(Q, K.T) / math.sqrt(Q.shape[-1]) # (n_stocks, n_stocks)
        weights = torch.softmax(scores, dim=1) # (n_stocks, n_stocks)
        A = torch.matmul(weights, V) # (n_stocks, hidden_size)
        s = torch.sigmoid(self.out(A)).squeeze(-1) # (n_stocks)

        return s

In [109]:
def portfolio_generators(s, cutoff = 4):
    """
    (n_stocks) -> (n_stocks)
    """
    n = s.shape[0]
    G = n // cutoff

    order = torch.argsort(s, descending=True)
    top_idx = order[:G]
    bottom_idx = order[-G:]

    bc = torch.zeros_like(s)

    top_scores = s[top_idx]
    bc[top_idx] = torch.softmax(top_scores, dim=0)

    bottom_scores = 1 - s[bottom_idx]
    bc[bottom_idx] = -torch.softmax(bottom_scores, dim=0)

    return bc

In [107]:
model1 = stockLSTM()
model2 = historyAttention()
model3 = crossAssetAttention()

In [110]:
def alphastock_forward(X):
    h = model1(X)
    r = model2(h)
    s = model3(r)
    bc = portfolio_generators(s)
    return bc

In [111]:
def portfolio_return(bc, price_ratio):
    return torch.sum(bc * price_ratio)

In [112]:
def sharpe_ratio(returns):
    mean = returns.mean()
    std = returns.std()

    if std < 1e-6:
        return torch.tensor(0.0)

    return mean/std

In [114]:
def train_step(X_seq, price_ratio_seq, optimizer, market_sharpe):
    returns = []
    for t in range(len(X_seq)):
        bc = alphastock_forward(X_seq[t])
        r_t = portfolio_return(bc, price_ratio_seq[t])
        returns.append(r_t)

    returns = torch.stack(returns)
    H = sharpe_ratio(returns)
    loss = -(H - market_sharpe)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return H.item()

In [118]:
optimizer = torch.optim.Adam(
    list(model1.parameters()) +
    list(model2.parameters()) +
    list(model3.parameters()),
    lr=1e-3
)